BƯỚC 1: CHUẨN BỊ DỮ LIỆU & RESHAPE

In [ ]:
import joblib
import numpy as np
import os
from google.colab import drive

# 1. Kết nối Drive (Thêm xử lý lỗi để tránh bị dừng)
try:
    drive.mount('/content/drive', force_remount=True)
except ValueError:
    print("⚠️ Drive có thể đã được mount. Đang tiếp tục...")

# 2. Cấu hình đường dẫn
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
DATA_PATH = os.path.join(BASE_DIR, "Binary_Data/")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "Models-CNN/") # Thư mục lưu CNN
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print("-" * 50)
print("🔄 BƯỚC 1: NẠP VÀ BIẾN ĐỔI CHIỀU DỮ LIỆU (RESHAPE)")

# 3. Load dữ liệu
print("   - Đang load dữ liệu từ Binary_Data...")
X_train = joblib.load(DATA_PATH + 'X_train.pkl')
y_train = joblib.load(DATA_PATH + 'y_train.pkl')
X_test = joblib.load(DATA_PATH + 'X_test.pkl')
y_test = joblib.load(DATA_PATH + 'y_test.pkl')

print(f"   - Shape gốc (X_train): {X_train.shape}")

# 4. Reshape cho CNN (Quan trọng)
# CNN 1D yêu cầu đầu vào 3 chiều: (Số mẫu, Số đặc trưng, 1 kênh)
print("   - Đang reshape dữ liệu...")
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"   - Shape mới (X_train_cnn): {X_train_cnn.shape}")
print(f"   - Shape mới (X_test_cnn) : {X_test_cnn.shape}")
print("✅ Đã xử lý xong dữ liệu cho CNN!")

BƯỚC 2: DỰNG MÔ HÌNH 1D-CNN (2 BLOCKS)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.optimizers import Adam

print("-" * 50)
print("🏗️ BƯỚC 2: ĐANG KHỞI TẠO KIẾN TRÚC 1D-CNN...")

def build_cnn_binary(input_shape):
    model = Sequential(name="CNN_Binary_NIDS")

    # --- BLOCK 1: KHAI PHÁ (32 filters) ---
    # Input shape: (37, 1)
    # Padding='same' để giữ nguyên chiều dài dữ liệu sau khi quét
    model.add(Conv1D(filters=32, kernel_size=3, padding='same', activation='relu', input_shape=input_shape))
    model.add(BatchNormalization()) # Ổn định dữ liệu
    model.add(MaxPooling1D(pool_size=2)) # Giảm kích thước còn một nửa

    # --- BLOCK 2: CHUYÊN SÂU (64 filters) ---
    model.add(Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling1D(pool_size=2)) # Giảm tiếp kích thước

    # --- CHUYỂN ĐỔI (FLATTEN) ---
    model.add(Flatten()) # Duỗi thẳng từ dạng khối (3D) sang dạng vector (1D)

    # --- LỚP PHÂN LOẠI (CLASSIFIER) ---
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5)) # Kỷ luật sắt: Tắt 50% nơ-ron để chống học vẹt

    # --- OUTPUT ---
    # 1 Nơ-ron + Sigmoid -> Trả về xác suất 0-1
    model.add(Dense(1, activation='sigmoid'))

    # Compile mô hình
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

    return model

# Khởi tạo mô hình
# X_train_cnn.shape[1:] chính là (37, 1)
model_cnn = build_cnn_binary(X_train_cnn.shape[1:])

# Hiện bảng thông số kỹ thuật
model_cnn.summary()

print("-" * 50)
print("✅ MÔ HÌNH ĐÃ SẴN SÀNG!")

In [ ]:
import joblib
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

print("-" * 50)
print("🚀 BƯỚC 3: THIẾT LẬP GIÁM SÁT VÀ BẮT ĐẦU TRAIN...")

# 1. Load lại Class Weights (để đảm bảo cân bằng)
class_weights = joblib.load(DATA_PATH + 'class_weights.pkl')
print(f"   - Đã load trọng số: {class_weights}")

# 2. Thiết lập Bộ giám sát (Callbacks)
# Lưu model vào thư mục Models-CNN
checkpoint_path = os.path.join(MODEL_SAVE_PATH, 'Scenario_CNN_Binary_Best.keras')

callbacks = [
    # Lưu học sinh giỏi nhất
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1),

    # Dừng nếu không tiến bộ sau 10 vòng
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),

    # Giảm tốc độ học nếu bị kẹt
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1)
]

# 3. Bắt đầu Train
print("\n🔥 START TRAINING CNN...")
history = model_cnn.fit(
    X_train_cnn, y_train,         # Dữ liệu 3 chiều
    validation_split=0.1,         # Cắt 10% làm bài kiểm tra thử
    epochs=50,                    # Số vòng tối đa
    batch_size=512,               # Học chậm mà chắc
    class_weight=class_weights,   # Áp dụng trọng số
    callbacks=callbacks,
    verbose=1
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

# 4. Vẽ biểu đồ kết quả
plt.figure(figsize=(14, 6))

# --- Biểu đồ Loss ---
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('CNN Loss (Hàm mất mát)')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

# --- Biểu đồ Accuracy ---
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc', color='green')
plt.plot(history.history['val_accuracy'], label='Val Acc', color='red')
plt.title('CNN Accuracy (Độ chính xác)')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.show()

In [ ]:
import joblib
import numpy as np
import os
from tensorflow.keras.models import load_model

print("-" * 50)
print("📂 BƯỚC 4.1: ĐANG NẠP DỮ LIỆU TEST...")

# 1. Cấu hình đường dẫn
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
DATA_PATH = os.path.join(BASE_DIR, "Binary_Data/")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "Models-CNN/")

# 2. Load dữ liệu Test
# Kiểm tra xem biến đã có sẵn trong RAM chưa để đỡ load lại
if 'X_test' not in globals():
    print("   - Đang đọc file từ đĩa...")
    X_test = joblib.load(DATA_PATH + 'X_test.pkl')
    y_test = joblib.load(DATA_PATH + 'y_test.pkl')
else:
    print("   - Dữ liệu đã có sẵn trong RAM.")

print(f"   - Số lượng mẫu Test: {len(y_test)}")
print("-" * 50)

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
import os

print("-" * 50)
print("🤖 BƯỚC 4.2: ĐANG DỰ ĐOÁN (PREDICTING)...")

# 1. Reshape dữ liệu Test (BẮT BUỘC CHO CNN)
# Dữ liệu gốc: (419297, 37) -> CNN cần: (419297, 37, 1)
print(f"   - Shape gốc: {X_test.shape}")
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
print(f"   - Shape sau khi Reshape: {X_test_cnn.shape}")

# 2. Load mô hình CNN Lite (Phiên bản tốt nhất đã lưu)
model_name = 'Scenario_CNN_Binary_Lite.keras'
model_path = os.path.join(MODEL_SAVE_PATH, model_name)

try:
    print(f"   - Đang tải mô hình từ: {model_path}")
    final_model = load_model(model_path)
    print("   ✅ Đã tải thành công model từ ổ cứng.")
except:
    print("   ⚠️ Không tìm thấy file trên ổ cứng. Đang sử dụng model trong RAM.")
    final_model = model_cnn # Biến model_cnn từ bước train trước

# 3. Thực hiện dự đoán
# Kết quả trả về là xác suất (VD: 0.99, 0.01, ...)
print("   - Đang tính toán xác suất...")
y_pred_prob = final_model.predict(X_test_cnn, verbose=1)

# 4. Chuyển đổi sang nhãn 0/1
# > 0.5 là Attack (1), ngược lại là Benign (0)
y_pred = (y_pred_prob > 0.5).astype("int32")

print("✅ ĐÃ DỰ ĐOÁN XONG!")
print("-" * 50)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("-" * 50)
print("📊 BƯỚC 4.3: BẢNG ĐIỂM CHI TIẾT & MA TRẬN NHẦM LẪN")

# 1. Tính toán điểm số chính thức
loss, accuracy = final_model.evaluate(X_test_cnn, y_test, verbose=0)

print(f"\n🎯 KẾT QUẢ TỔNG QUÁT (CNN):")
print(f"   - Test Accuracy : {accuracy * 100:.2f}%")
print(f"   - Test Loss     : {loss:.4f}")

# 2. Báo cáo chi tiết (Precision, Recall, F1)
print("\n📝 BÁO CÁO CHI TIẾT TỪNG LỚP:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Attack (1)']))

# 3. Vẽ Ma trận nhầm lẫn (Confusion Matrix)
print("heatmap Ma trận nhầm lẫn:")
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
# Em dùng màu ĐỎ (Reds) để phân biệt với mô hình DNN (màu Xanh) trước đó
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'])
plt.title('Confusion Matrix - CNN Model')
plt.ylabel('Thực tế (Actual)')
plt.xlabel('Dự đoán (Predicted)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

print("-" * 50)
print("📈 BƯỚC 4.4: VẼ BIỂU ĐỒ ROC & AUC...")

# 1. Tính toán FPR, TPR và các ngưỡng
# y_test: Nhãn thực tế (0 hoặc 1)
# y_pred_prob: Xác suất dự đoán (từ 0.0 đến 1.0) - Đã có ở bước 4.2
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

# 2. Tính diện tích dưới đường cong (AUC)
roc_auc = auc(fpr, tpr)
print(f"   - Chỉ số AUC: {roc_auc:.4f} (Càng gần 1 càng tốt)")

# 3. Vẽ biểu đồ
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Đường chéo ngẫu nhiên (50%)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Tỷ lệ Báo động giả)')
plt.ylabel('True Positive Rate (Độ nhạy - Recall)')
plt.title('Receiver Operating Characteristic (ROC) - CNN Binary')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
import time
import numpy as np

def calculate_inference_time(model, X_data, num_samples=1000):
    # Lấy ngẫu nhiên num_samples mẫu để test
    indices = np.random.choice(len(X_data), num_samples, replace=False)
    X_sample = X_data[indices]

    # Warm-up (chạy nháp vài lần để GPU/CPU nóng máy)
    _ = model.predict(X_sample[:10], verbose=0)

    # Bắt đầu đo
    start_time = time.time()
    _ = model.predict(X_sample, verbose=0)
    end_time = time.time()

    # Tính toán
    total_time = end_time - start_time
    time_per_sample = (total_time / num_samples) * 1_000_000 # Đổi sang microseconds (µs)

    return time_per_sample

print("-" * 50)
print("⏱️ ĐANG ĐO TỐC ĐỘ XỬ LÝ (INFERENCE TIME)...")

# 1. Đo DNN (Giả sử model_dnn là biến lưu mô hình DNN của anh)
# Lưu ý: X_test dùng cho DNN là mảng 2 chiều
try:
    dnn_time = calculate_inference_time(model_dnn, X_test, 1000)
    print(f"   - DNN Speed: {dnn_time:.2f} µs/gói tin")
except NameError:
    print("   - Không tìm thấy biến model_dnn trong RAM (Cần load lại nếu đã tắt session)")
    dnn_time = 0 # Giá trị giả định để điền bảng

# 2. Đo 1D-CNN (Giả sử final_model là biến lưu mô hình CNN)
# Lưu ý: X_test_cnn dùng cho CNN là mảng 3 chiều
try:
    cnn_time = calculate_inference_time(final_model, X_test_cnn, 1000)
    print(f"   - 1D-CNN Speed: {cnn_time:.2f} µs/gói tin")
except NameError:
    print("   - Không tìm thấy biến final_model (CNN) trong RAM")
    cnn_time = 0

print("-" * 50)